# Dunnhumby seed 43 — N·V 공유 표현 M2 / M5 개발 실험
N과 V를 **둘 다** 사용합니다. 상품별 자유 N/V 벡터와 개인이력 평균을, 사용자·상품 속성에 적용하는 공유 선형변환으로 바꿉니다. 입력은 기존 historical q_N·q_V, 상품 구매자 q_N의 축소평균, 상품 단가의 축소평균 백분위입니다. 세 RBF 기저(중심 0/0.5/1, 폭 .25) → 축별 4차원 변환이며 추가 파라미터는 48개입니다. 상품 N은 item CLV가 아니라 구매자 구성 속성입니다.

점수 = ID 내적 + α_N × N표현 내적 + α_V × V표현 내적. ID만 binary LightGCN 2층을 통과하고, N/V 표현은 전파 후 연결합니다. 모든 표현이 **하나의 optimizer에서 공동학습**됩니다. M2는 무가중 BPR, M5는 수정 원형 M4(λ=.25)의 고정 행 가중 BPR입니다. 동결·외부 보정·재정렬 없음. M2에는 q_C를 쓰지 않으며 CLV **구성요소** 표현입니다.

첫 실행: α_N=α_V=.05, ID64, L2=.001, K=1 uniform, seed43. N/V 강도는 분리되어 있지만 데이터별 최적값을 이미 찾았다는 뜻은 아닙니다. α는 최종 점수 기여율이 아니므로 실제 N/V 점수와 gradient도 저장합니다.

신규상품 개발과업(학습≤683, 개발684–690), MIN_ITEM_INTER=1. 학습 양성 상품은 해당 구매자 기여를 빼고, 다른 관측이 없으면 해당 축만 0으로 둡니다. 사용자 CLV와 전체 학습 가격 참조분포는 고정합니다. 카탈로그는 삭제하지 않습니다.

**M1·수정 원형 M4는 이전 결과를 재사용합니다. 새 학습은 M2·M5 두 개뿐입니다.** 최대300 epoch, 25마다 평가, 전체 가격·구매금액 가중 적중값@10으로 선택, 100 이후 4회 미개선 시 중단. 판독은 전체 경제지표 두 개 @10의 M1/M4 대비 개선 및 여섯 Recall/NDCG 각 M1의99% 보호입니다. 고CLV는 보조 분석. 반복 노출 개발시드이므로 유의성·일반화·CLV 귀속을 주장하지 않습니다. test/holdout 없음.

검증 범위: 로컬 CPU 합성 학습·재개·캐시 점검. 이 노트북의 Drive 연결과 전체 GPU 학습은 Colab에서 실행해야 합니다.

In [ ]:
from pathlib import Path
import os, sys, subprocess, json
from google.colab import drive
ROOT = Path('/content/drive/MyDrive/논문/data')
REPORT = ROOT/'results_v3_dunnhumby_original_m4_lambda025_seed43_v1/reports/result.json'
if not REPORT.is_file():
    if os.path.ismount('/content/drive'):
        raise RuntimeError('Drive 연결은 되어 있으나 기존 λ=.25 결과가 없습니다. 계정/REPORT 경로를 확인하세요. 새 학습 안 함.')
    try:
        drive.mount('/content/drive')
    except (ValueError, NotImplementedError) as exc:
        raise RuntimeError('Drive 연결 실패. 학습은 시작되지 않았습니다. 연결 계정을 확인하세요.') from exc
    if not REPORT.is_file():
        raise RuntimeError('기존 λ=.25 결과가 없습니다. 재학습하지 않고 중단합니다.')
SOURCE_COMMIT = '49d8ab94284c116f8691bb0416bf5671034cf63d'
REPO = Path('/content/clv-shared-nv-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
if 'lightgcn_clv_v3' in sys.modules:
    assert Path(sys.modules['lightgcn_clv_v3'].__file__).resolve().parent == REPO.resolve(), '다른 소스가 로드되어 있습니다. 런타임을 다시 시작하세요.'
os.chdir(REPO)
sys.path.insert(0, str(REPO))
import clv_shared_nv_feature_screen as screen
import pandas as pd
OUT = ROOT/'results_v3_dunnhumby_shared_nv_features_seed43_v1'
print('고정 소스:', SOURCE_COMMIT, '| 실행:', screen.VERSION)

## 1. 입력·기존 결과 검증 (학습 없음)
기존 결과 SHA·체크포인트·선택규칙을 검증합니다. 학습에서 자기 기여를 뺀 N/V 입력과 무효 입력 M4 가중치도 확인합니다. 누락되면 새 학습 전에 중단합니다.

In [ ]:
ALPHA_N = 0.05
ALPHA_V = 0.05
cfg, prepared, audit = screen.prepare(REPORT, OUT, alpha_n=ALPHA_N, alpha_v=ALPHA_V)
assert cfg.seeds == (43,) and cfg.epochs == 300 and cfg.positive_weight_lambda == .25
print('재사용:', [a['model_id'] for a in prepared['anchors']])
print('새 학습:', [s['model_id'] for s in screen.specs(prepared['feature_settings'])])
print(audit[audit.group.isin(['all','any_input_invalid'])].to_string(index=False))

## 2. M2·M5 학습
완료 epoch마다 optimizer·난수상태를 Drive에 저장합니다. 중단 후 같은 설정으로 다시 실행하면 이어지고, 완료 모델은 건너뜁니다. 런타임 자동 재연결을 보장하는 기능은 아닙니다. 기존 M1/M4는 학습하지 않습니다.

In [ ]:
import torch
assert torch.cuda.is_available(), '전체 실험은 GPU 런타임을 사용하세요.'
paths = screen.run(cfg, prepared)
print(json.dumps(paths, ensure_ascii=False, indent=2))

## 3. 전체 지표 미리보기·원본 ZIP
M2 완료 후 M5가 진행 중이어도 저장된 reports 폴더를 읽을 수 있습니다. 미리보기만으로 판정하지 말고 ZIP의 전체·세그먼트 지표, 비교, 곡선, 작동 진단을 함께 확인합니다.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
report = json.loads((OUT/'reports/result.json').read_text())
paths = report['paths']
absolute = pd.read_csv(paths['absolute'])
metrics = list(screen.base.ACCURACY) + ['price_purchase_amount_weighted_hit@10','vndcg@10','price_purchase_amount_weighted_hit@20','vndcg@20','price_purchase_amount_weighted_hit@50','vndcg@50','coverage@10','user_value_tendency_recommended_price_alignment']
print(absolute[['model_id','selected_epoch','stopped_epoch']+metrics].set_index('model_id').T.to_string())
print(json.dumps(report['reading'], ensure_ascii=False, indent=2))
print('N/V 작동 진단:', paths['diagnostics'])
zip_path = Path('/content/shared_nv_m2_m5_seed43_results.zip')
with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as archive:
    for path in paths.values():
        archive.write(path, arcname=Path(path).name)
    for name in ('m4_validity_audit.csv', 'feature_diagnostic.json'):
        archive.write(OUT/name, arcname=name)
files.download(str(zip_path))